In [ ]:
!pip install -q tiktoken

In [ ]:
from pathlib import Path
import os
import sys
import math
import torch

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/arjunbhattarai123/"
    "gpt-2-llm-from-scratch-training-project-1"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project not found: {PROJECT_ROOT}")

# Training targets
START_STEP = 500_000
MAX_STEPS = 650_000

# The full learning-rate schedule spans 500K -> 800K.
TOTAL_SCHEDULE_STEPS = 300_000

assert MAX_STEPS - START_STEP == 150_000

# Locate original 500K checkpoint
candidates = [
    Path("/kaggle/working/gpt2_wikitext_500k.pt"),
    PROJECT_ROOT / "gpt2_wikitext_500k.pt",
    PROJECT_ROOT / "checkpoints/gpt2_wikitext_500k.pt",
    Path(
        "/kaggle/input/datasets/arjunbhattarai123/"
        "gpt2-wikitext-500k/gpt2_wikitext_500k.pt"
    ),
]

if not any(p.is_file() for p in candidates):
    candidates += list(
        Path("/kaggle/input").rglob("gpt2_wikitext_500k.pt")
    )

CHECKPOINT_PATH = next(
    (p for p in candidates if p.is_file()), None
)

if CHECKPOINT_PATH is None:
    raise FileNotFoundError("500K checkpoint not found.")

OUTPUT_CHECKPOINT = Path(
    "/kaggle/working/gpt2_wikitext_650k.pt"
)

BEST_CHECKPOINT = Path(
    "/kaggle/working/gpt2_wikitext_650k_best.pt"
)

METRICS_PATH = Path(
    "/kaggle/working/training_metrics_500k_to_650k.csv"
)

RESUME_CHECKPOINT_PATH = Path(
    "/kaggle/working/gpt2_wikitext_500k_fresh_scheduler.pt"
)

# Load original checkpoint without modifying it.
checkpoint_meta = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

resume_step = int(checkpoint_meta["global_step"])

if resume_step != START_STEP:
    raise ValueError(
        f"Expected 500K checkpoint, found {resume_step:,}"
    )

if "model_state_dict" not in checkpoint_meta:
    raise KeyError("Model state missing.")

if "optimizer_state_dict" not in checkpoint_meta:
    raise KeyError("Optimizer state missing.")

saved_optimizer = checkpoint_meta["optimizer_state_dict"]

print(
    "Saved optimizer LR:",
    [g["lr"] for g in saved_optimizer["param_groups"]]
)

# Start a fresh 300K-step scheduler.
# Preserve model, optimizer, RNG, and progress.
checkpoint_meta["scheduler_state_dict"] = None

torch.save(
    checkpoint_meta,
    RESUME_CHECKPOINT_PATH
)

# Project environment
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if not torch.cuda.is_available():
    raise RuntimeError("Please enable GPU in Kaggle settings.")

print("=" * 55)
print("GPT-2 SESSION 1: 500K -> 650K")
print("=" * 55)
print("Original checkpoint:", CHECKPOINT_PATH)
print("Resume checkpoint:", RESUME_CHECKPOINT_PATH)
print("Output checkpoint:", OUTPUT_CHECKPOINT)
print("Starting step:", START_STEP)
print("Target step:", MAX_STEPS)
print("Additional steps:", MAX_STEPS - START_STEP)
print("GPU:", torch.cuda.get_device_name(0))
print("=" * 55)

In [ ]:
print("Resume:", RESUME_CHECKPOINT_PATH)
print("Output:", OUTPUT_CHECKPOINT)

In [ ]:
required_files = [
    PROJECT_ROOT / "src/llm/gpt/model.py",
    PROJECT_ROOT / "src/llm/training/trainer.py",
    PROJECT_ROOT / "src/llm/training/checkpoint.py",
    PROJECT_ROOT / "data/wikitext-103/train.txt",
    PROJECT_ROOT / "data/wikitext-103/validation.txt",
    RESUME_CHECKPOINT_PATH,
]

missing = [str(p) for p in required_files if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Missing files:\n" + "\n".join(missing)
    )

print("All required files found.")

In [ ]:
train_path = PROJECT_ROOT / "data/wikitext-103/train.txt"
validation_path = PROJECT_ROOT / "data/wikitext-103/validation.txt"

train_text = train_path.read_text(encoding="utf-8")
validation_text = validation_path.read_text(encoding="utf-8")

print("Training characters:", len(train_text))
print("Validation characters:", len(validation_text))

In [ ]:
import tiktoken

from llm.preprocessing.dataset import create_data_loader

tokenizer = tiktoken.get_encoding("gpt2")

BATCH_SIZE = 1
CONTEXT_LENGTH = 1024
NUM_WORKERS = 2

train_loader = create_data_loader(
    text=train_text,
    batch_size=BATCH_SIZE,
    max_length=CONTEXT_LENGTH,
    stride=CONTEXT_LENGTH,
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
)

validation_loader = create_data_loader(
    text=validation_text,
    batch_size=BATCH_SIZE,
    max_length=CONTEXT_LENGTH,
    stride=CONTEXT_LENGTH,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

In [ ]:
from llm.gpt.config import GPT_CONFIG_124M
from llm.gpt.model import GPTModel

config = GPT_CONFIG_124M.copy()
config["context_length"] = CONTEXT_LENGTH

model = GPTModel(config).to(device)

saved_optimizer = checkpoint_meta["optimizer_state_dict"]
saved_groups = saved_optimizer["param_groups"]

all_params = [
    p for p in model.parameters() if p.requires_grad
]

decay_params = [
    p for p in model.parameters()
    if p.requires_grad and p.ndim >= 2
]

no_decay_params = [
    p for p in model.parameters()
    if p.requires_grad and p.ndim < 2
]

if len(saved_groups) == 1:
    if len(saved_groups[0]["params"]) != len(all_params):
        raise ValueError("Optimizer parameter count mismatch.")

    optimizer_param_groups = [{"params": all_params}]

elif len(saved_groups) == 2:
    if (
        len(saved_groups[0]["params"]) != len(decay_params)
        or len(saved_groups[1]["params"]) != len(no_decay_params)
    ):
        raise ValueError("Optimizer grouping mismatch.")

    optimizer_param_groups = [
        {"params": decay_params},
        {"params": no_decay_params},
    ]

else:
    raise ValueError("Unsupported optimizer group structure.")

first_group = saved_groups[0]

optimizer = torch.optim.AdamW(
    optimizer_param_groups,
    lr=float(first_group["lr"]),
    betas=tuple(first_group.get("betas", (0.9, 0.999))),
    eps=float(first_group.get("eps", 1e-8)),
    weight_decay=float(first_group.get("weight_decay", 0.01)),
)

for new_group, old_group in zip(optimizer.param_groups, saved_groups):
    new_group["weight_decay"] = float(
        old_group.get("weight_decay", 0.0)
    )

# Learning-rate schedule:
# 500K -> 800K, not 500K -> 650K.
BASE_LR = float(first_group["lr"])
MIN_LR = 1e-7

if not (0 < MIN_LR < BASE_LR):
    raise ValueError(
        f"Invalid LR: BASE_LR={BASE_LR}, MIN_LR={MIN_LR}"
    )

def lr_lambda(step):
    progress = min(
        max(step / TOTAL_SCHEDULE_STEPS, 0.0),
        1.0
    )

    cosine = 0.5 * (
        1.0 + math.cos(math.pi * progress)
    )

    lr = MIN_LR + (BASE_LR - MIN_LR) * cosine

    return lr / BASE_LR

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lr_lambda
)

print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Base LR:", BASE_LR)
print("Minimum LR:", MIN_LR)
print("Schedule length:", TOTAL_SCHEDULE_STEPS)
print("Session target:", MAX_STEPS)

In [ ]:
from llm.training.trainer import train_model_simple
from llm.training.checkpoint import load_checkpoint

print("Training functions imported successfully.")

In [ ]:
# Do not restore the old scheduler.
preflight = load_checkpoint(
    model=model,
    optimizer=optimizer,
    scaler=None,
    path=str(RESUME_CHECKPOINT_PATH),
    device=device,
    scheduler=None,
    restore_rng=False,
)

assert int(preflight["global_step"]) == 500_000

assert preflight.get("scheduler_state_dict") is None

print("Checkpoint compatibility: PASSED")
print("Global step:", preflight["global_step"])
print("Optimizer LR:", [g["lr"] for g in optimizer.param_groups])
print("Fresh scheduler:", "READY")

In [ ]:
import time

EVAL_FREQ = 2_500
EVAL_ITER = 20

print("=" * 60)
print("STARTING GPT-2 TRAINING")
print("500K -> 650K")
print("=" * 60)

start_time = time.time()

train_losses, validation_losses, tokens_seen = train_model_simple(
    model=model,
    train_loader=train_loader,
    val_loader=validation_loader,
    optimizer=optimizer,
    device=device,

    # Safety ceiling; max_steps controls the actual stopping point.
    num_epochs=100,

    eval_freq=EVAL_FREQ,
    eval_iter=EVAL_ITER,

    start_context="The history of the world",
    tokenizer=tokenizer,

    checkpoint_path=str(OUTPUT_CHECKPOINT),
    config=config,

    # Final global step, not additional steps.
    max_steps=MAX_STEPS,

    resume_from_checkpoint=str(RESUME_CHECKPOINT_PATH),

    scheduler=scheduler,

    gradient_accumulation_steps=1,

    metrics_path=str(METRICS_PATH),

    periodic_checkpoint_steps=5_000,

    best_checkpoint_path=str(BEST_CHECKPOINT),

    eval_generation=False,
)

elapsed_hours = (time.time() - start_time) / 3600

print("=" * 60)
print("SESSION 1 TRAINING FINISHED")
print("Elapsed hours:", round(elapsed_hours, 2))
print("=" * 60)

In [ ]:
if not OUTPUT_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Checkpoint missing: {OUTPUT_CHECKPOINT}"
    )

final_checkpoint = torch.load(
    OUTPUT_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

final_step = int(final_checkpoint["global_step"])

print("=" * 60)
print("SESSION 1 FINAL REPORT")
print("=" * 60)

print("Checkpoint:", OUTPUT_CHECKPOINT)
print("Global step:", final_step)
print("Epoch:", final_checkpoint.get("epoch"))
print("Training loss:", final_checkpoint.get("train_loss"))
print("Validation loss:", final_checkpoint.get("val_loss"))
print("Best validation loss:", final_checkpoint.get("best_val_loss"))

print(
    "Optimizer LR:",
    [
        group["lr"]
        for group in final_checkpoint["optimizer_state_dict"]["param_groups"]
    ]
)

print(
    "Scheduler state saved:",
    final_checkpoint.get("scheduler_state_dict") is not None
)

print("Metrics CSV exists:", METRICS_PATH.exists())

if final_step == 650_000:
    print("SUCCESS: 650K STEPS COMPLETED.")
else:
    print(
        f"Training stopped at {final_step:,}. "
        "Resume from the latest valid checkpoint."
    )

if final_step == 650_000:
    assert final_checkpoint.get("scheduler_state_dict") is not None, (
        "Scheduler state is missing. Do not start Session 2 "
        "until scheduler continuity is resolved."
    )